# Symbols

Turning what a reader typed into a symbol that exists. Pure: takes the instruments frame as an
argument, imports nothing. Callers: `%run ./symbols`.

**It never near-matches** - no edit distance, no closest guess. Answering about the wrong instrument
is worse than failing, so anything ambiguous raises and tells the model to call `list_instruments`
instead of guessing a ticker. A fragment that prefixes exactly one instrument name does resolve; that
is a unique match, not a near one.

In [ ]:
class UnknownSymbolError(ValueError):
    """Raised when nothing resolves. The message is written for the model, not for a log."""


# What a reader types -> the canonical ticker. Data, not logic; extend it freely, under one rule:
# an alias names the SAME instrument, never a proxy for it. SPX -> SPY would break that - SPX is the
# S&P 500 index and SPY is an ETF tracking it, with a different price, dividends and tracking error -
# and so would S&P 500 -> SPY, which is the same substitution wearing a friendlier name. Neither is
# here. A reader asking about the index gets a miss, sees SPY described as an ETF in
# list_instruments, and substitutes knowingly; the resolver does not decide that quietly.
ALIASES = {
    "APPLE": "AAPL",
    "MICROSOFT": "MSFT",
    "GOOGLE": "GOOG",      # class C; if GOOGL is ever ingested this picks one share class
    "ALPHABET": "GOOG",
    "DOW": "^DJI",
    "DOW JONES": "^DJI",
    "NASDAQ": "^IXIC",     # the Composite, not the Nasdaq-100
    "BITCOIN": "BTC-USD",
    "BTC": "BTC-USD",
    "ETHEREUM": "ETH-USD",
    "ETH": "ETH-USD",
    "EUR/USD": "EURUSD=X",
    "EURUSD": "EURUSD=X",
}


def resolve_symbol(frame, symbol):
    """`frame` is the instruments frame; returns a symbol that is in it, or raises.

    Order: the literal uppercase, the alias table, `{X}-USD`, `{X}=X`, then a *unique* prefix match
    against the instrument names. Every candidate is checked against the frame, so a symbol that
    comes back always has data behind it - an alias pointing at an instrument nobody ingested raises
    like any other miss.

    Two consequences of that order worth knowing, because both look like near-matching and are not:
    a fragment resolves when it prefixes exactly one name, so "APPL" and "Apple Inc" both reach
    AAPL; and an alias is an explicit human decision, so it wins even where the names alone would be
    ambiguous - with GOOG and GOOGL both present, "ALPHABET" is GOOG by the table, while "Alphabet
    Inc" prefixes both names and therefore raises.
    """
    if not isinstance(symbol, str) or not symbol.strip():
        raise UnknownSymbolError(
            "No instrument was named. Call list_instruments to see what is available."
        )

    known = set(frame["symbol"])
    raw = " ".join(symbol.strip().upper().split())
    squashed = raw.replace(" ", "")

    for candidate in (raw, ALIASES.get(raw), ALIASES.get(squashed),
                      f"{squashed}-USD", f"{squashed}=X"):
        if candidate in known:
            return candidate

    prefixed = sorted({s for s, name in zip(frame["symbol"], frame["name"])
                       if isinstance(name, str) and name.upper().startswith(raw)})
    if len(prefixed) == 1:
        return prefixed[0]

    ambiguous = f" {len(prefixed)} instruments start with it: {prefixed}." if prefixed else ""
    raise UnknownSymbolError(
        f"'{symbol}' is not an instrument I have data for.{ambiguous}"
        " Call list_instruments to see what is available, then use one of those symbols exactly."
        " Do not guess a ticker."
    )